In [9]:
import pandas as pd
import sys
sys.path.append("../src")
from data_prep import clean_audio_dataset, build_catalog, add_genre_label, match_my_tracks

In [5]:
tracks = pd.read_csv("../data/processed/my_taste_scores.csv")
kaggle = pd.read_csv("../data/processed/clean_tracks.csv")
multilingual = pd.read_csv("../data/raw/spotify_tracks.csv")


In [6]:
# clean each one using the same function
kaggle_clean = clean_audio_dataset(kaggle, "track_name", "artists", id_col="track_id")
multilingual_clean = clean_audio_dataset(multilingual, "track_name", "artist_name", id_col="track_id")

In [7]:
kaggle_clean = add_genre_label(kaggle_clean, source_col="track_genre", label_type="genre")
multilingual_clean = add_genre_label(multilingual_clean, source_col="language", label_type="language")

print("kaggle_clean columns:", kaggle_clean.columns.tolist())
print("multilingual_clean columns:", multilingual_clean.columns.tolist())

kaggle_clean columns: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'genre_or_language', 'label_type']
multilingual_clean columns: ['track_id', 'track_name', 'artists', 'year', 'popularity', 'artwork_url', 'album_name', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence', 'track_url', 'language', 'genre_or_language', 'label_type']


In [11]:
def load_and_prepare_indian_file(filepath,language_name):
    """Load one language CSV, rename columns to match our pipeline, tag with its language."""
    df=pd.read_csv(filepath)
    df=df.rename(columns={
        "song_name":"track_name",
        "singer":"artists",
        "Valence":"valence"
    })

    def duration_to_ms(d):
        try:
            mins, secs = map(int, d.split(":"))
            return (mins * 60 + secs) * 1000
        except:
            return None
    
    df["duration_ms"] = df["duration"].apply(duration_to_ms)
    df["language"] = language_name
    
    return df

In [12]:
import glob
import os
folder ="../data/raw/indian_languages/"
files = glob.glob(folder + "*.csv")

all_indian_dfs = []
for filepath in files:
    filename = os.path.basename(filepath)
    language_name = filename.replace("_songs.csv", "").replace("_Songs.csv", "")
    df = load_and_prepare_indian_file(filepath, language_name)
    all_indian_dfs.append(df)
    print(f"{language_name}: {df.shape}")

indian_combined = pd.concat(all_indian_dfs, ignore_index=True)
print("\nTotal combined:", indian_combined.shape)

Marathi: (4699, 20)
Gujarati: (2115, 20)
Odia: (940, 20)
Tamil: (4677, 20)
Rajasthani: (541, 20)
Bengali: (958, 20)
Bhojpuri: (519, 20)
Malayalam: (479, 20)
Punjabi: (3818, 20)
Telugu: (4996, 20)
Old: (2448, 20)
Haryanvi: (228, 20)
Hindi: (1184, 20)
Kannada: (3559, 20)
Urdu: (3116, 20)

Total combined: (34277, 20)


In [16]:
indian_clean = clean_audio_dataset(indian_combined, "track_name", "artists", id_col=None)
indian_clean = add_genre_label(indian_clean, source_col="language", label_type="language")
indian_clean["instrumentalness"] = 0.0  # not present in this dataset; filled with neutral default

print("indian_clean:", indian_clean.shape)
print(indian_clean.columns.tolist())

indian_clean: (28788, 23)
['track_name', 'artists', 'singer_id', 'duration', 'language', 'released_date', 'danceability', 'acousticness', 'energy', 'liveness', 'loudness', 'speechiness', 'tempo', 'mode', 'key', 'valence', 'time_signature', 'popularity', 'Stream', 'duration_ms', 'genre_or_language', 'label_type', 'instrumentalness']


In [ ]:
common_cols = ["track_name", "artists", "danceability", "energy", "loudness",
               "speechiness", "acousticness", "instrumentalness", "liveness",
               "valence", "tempo", "duration_ms", "genre_or_language", "label_type"]

catalog = build_catalog([kaggle_clean, multilingual_clean, indian_clean], common_cols)
catalog.to_csv("../data/processed/full_catalog.csv", index=False)
print("combined catalog:", catalog.shape)
print(catalog["label_type"].value_counts())

In [10]:
print("combined catalog:", catalog.shape)

combined catalog: (121923, 15)


In [11]:
matched = match_my_tracks(tracks, catalog)

Your songs: 1553
Matched: 206
Match rate: 13.3%


In [12]:
catalog.to_csv("../data/processed/full_catalog.csv", index=False)
print("saved",catalog.shape)

saved (121923, 15)


In [14]:
print(catalog.shape)
print(catalog["label_type"].value_counts())

(121923, 15)
label_type
genre       79921
language    42002
Name: count, dtype: int64


In [15]:
print(catalog["genre_or_language"].value_counts().tail(10))

genre_or_language
alternative    238
jazz           214
metal          211
punk           207
Malayalam      179
rock           167
reggae         165
house          134
indie          107
reggaeton       62
Name: count, dtype: int64


In [17]:
print(catalog["genre_or_language"].value_counts().head(20))

genre_or_language
English        13824
Tamil           9659
Unknown         9186
Korean          5099
Hindi           3797
study            996
heavy-metal      983
bluegrass        968
forro            967
malay            960
black-metal      955
cantopop         951
happy            948
club             941
afrobeat         941
comedy           940
acoustic         940
breakbeat        936
anime            936
tango            934
Name: count, dtype: int64


In [13]:
tracks = pd.read_csv("../data/processed/my_taste_scores.csv")
print(tracks.columns.tolist())

['track', 'plays', 'avg_completion', 'skip_rate', 'score']


In [ ]:
# 1. for each of your unmatched songs, find the closest-sounding title in the catalog
# 2. only accept matches above a similarity threshold (to avoid wrong matches)
# 3. combine these fuzzy matches with your existing exact matches

In [25]:
# build what's needed for fuzzy matching
matched_tracks = set(matched["track"])
unmatched = tracks[~tracks["track"].isin(matched_tracks)].copy()
print("still unmatched:", len(unmatched))

catalog_keys = catalog["track_key"].tolist()

still unmatched: 1347


In [26]:
from rapidfuzz import process, fuzz

fuzzy_results = []
for idx, track in enumerate(unmatched["track"]):
    best_match = process.extractOne(track, catalog_keys, scorer=fuzz.token_sort_ratio)
    if best_match and best_match[1] >= 88:  # raised from 85
        matched_key = best_match[0]
        # extract just the title part (before " - ") from both sides
        my_title = track.split(" - ")[0].lower().strip()
        their_title = matched_key.split(" - ")[0].lower().strip()
        title_similarity = fuzz.ratio(my_title, their_title)
        
        # only accept if titles themselves are genuinely close (not just artist match)
        if title_similarity >= 80:
            fuzzy_results.append({"track": track, "matched_key": matched_key, 
                                   "similarity": best_match[1], "title_similarity": title_similarity})

fuzzy_df = pd.DataFrame(fuzzy_results)
print("fuzzy matches found:", len(fuzzy_df))
fuzzy_df.sort_values("title_similarity").head(15)

fuzzy matches found: 41


,track,matched_key,similarity,title_similarity
37,Piya Tose Naina Laage Re Unplugged - Jonita Ga...,Piya Tose Naina Laage Re - Unplugged - Jonita ...,98.039216,82.758621
22,Tera Mera Rishta 2.0 - Mustafa Zahid,Tera Mera Rishta - Mustafa Zahid,94.117647,88.888889
0,O Saathi - Atif Aslam,O Saathi - Atif Aslam;Arko,89.361702,100.000000
23,"Saami Saami (From ""Pushpa The Rise Part - 01"")...","Saami Saami (From ""Pushpa The Rise Part - 01"")...",88.888889,100.000000
24,Teri Mitti - B Praak,Teri Mitti - Arko;B Praak,88.888889,100.000000
25,Ilzaam (From the Album 'Industry') - Arjun Kan...,Ilzaam (From the Album 'Industry') - Arjun Kan...,95.238095,100.000000
26,"Lagdi Lahore Di (From ""Street Dancer 3D"") - Gu...","Lagdi Lahore Di (From ""Street Dancer 3D"") - Gu...",90.476190,100.000000
27,Dil Ibaadat - Pritam,Dil Ibaadat - Pritam;KK,93.023256,100.000000
28,"Agar Tum Saath Ho (From ""Tamasha"") - Alka Yagnik","Agar Tum Saath Ho (From ""Tamasha"") - Alka Yagn...",88.073394,100.000000
29,Pehli Nazar Mein - Atif Aslam,Pehli Nazar Mein - Atif Aslam;Pritam,89.230769,100.000000


In [27]:
print(f"Exact matches: {len(matched)}")
print(f"Fuzzy matches added: {len(fuzzy_df)}")
print(f"Total: {len(matched) + len(fuzzy_df)}")
print(f"New match rate: {(len(matched) + len(fuzzy_df))/len(tracks)*100:.1f}%")

Exact matches: 206
Fuzzy matches added: 41
Total: 247
New match rate: 15.9%


In [28]:
fuzzy_matched = fuzzy_df.merge(catalog, left_on="matched_key", right_on="track_key", how="inner")
fuzzy_matched = fuzzy_matched.merge(tracks, on="track", how="left")

all_matched = pd.concat([matched, fuzzy_matched], ignore_index=True)
all_matched = all_matched.drop_duplicates(subset="track")

print(f"Final total matched: {len(all_matched)}")
print(f"Final match rate: {len(all_matched)/len(tracks)*100:.1f}%")

all_matched.to_csv("../data/processed/my_matched_songs.csv", index=False)

Final total matched: 247
Final match rate: 15.9%


we added new dataset to increase accuracy
